# [title] One-state model: simulator and exact reference

**Question.** Before any neural network is trained: do we have a simulator of the
one-state motor-adaptation model and an exact likelihood for it that we can trust as the
reference for the learned likelihood?

**Scope.** [#12](https://github.com/opherdonchin/MotorAdpatationSBI/issues/12), PR 1.
The model, priors and schedules are fixed by
[decision 0002](../docs/decisions/0002-one-state-model.md).

**Agreed checks** (numbering as in #12):

1. *Filter against brute force:* the Kalman log-likelihood equals the log density of the
   exact joint Gaussian of the responses.
2. *Simulator against the model:* simulated sittings have the mean and covariance of that
   same joint Gaussian.
3. *Prior predictive:* sittings drawn from the prior, for Opher to judge.
4. *Exact recovery and calibration:* posteriors sampled with the Kalman likelihood cover
   the true values, and simulation-based calibration (SBC) ranks are uniform.

**Words.** A *sitting* is one simulated session of one subject: a perturbation schedule
and the response on every trial. The likelihood is for a whole sitting.

In [ ]:
# [imports]
import logging
import subprocess
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor
import pytensor.tensor as pt
import xarray as xr
from scipy.linalg import solve_triangular
from scipy.special import expit, logit
from scipy.stats import chi2, kstest, multivariate_normal, norm

In [ ]:
# [config] Root seed, prior constants, and the size of every check
ROOT_SEED = 293292578773399405276523676509063398222  # secrets.randbits(128)

REPO = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
GIT_COMMIT = subprocess.check_output(["git", "describe", "--always", "--dirty"], text=True).strip()
OUT_DIR = REPO / "analyses" / "sim" / "one_state"

# Parameter order used everywhere: theta = (A, B, sigma_x, sigma_y).
# Everything is in units of the perturbation size.
PERTURBATION_VALUES = np.array([0.0, 1.0, -1.0])
N_BLOCKS_MIN, N_BLOCKS_MAX = 3, 10

# Each prior is normal on an unbounded scale; the stated range is its central 95% interval.
ranges = np.array([
    logit([0.75, 0.999]),   # A
    logit([0.01, 0.5]),     # B
    np.log([1 / 15, 1 / 3]),  # sigma_y
    np.log([1 / 15, 1 / 5]),  # sigma_x / sigma_y
    np.log([10, 50]),       # block length in trials
])
mu = ranges.mean(axis=1)
sigma = np.ptp(ranges, axis=1) / (2 * norm.ppf(0.975))
mu_logit_A, mu_logit_B, mu_log_sigma_y, mu_log_ratio, mu_log_block_len = mu
sigma_logit_A, sigma_logit_B, sigma_log_sigma_y, sigma_log_ratio, sigma_log_block_len = sigma

# Check 1: filter against brute force
N_FILTER_CASES = 50
T_SHORT = 8
# Check 2: simulator against the model
N_SIM_CASES = 40
N_SIM = 20_000
# Check 3: prior predictive
N_PRIOR_PRED = 12
# Check 4: exact recovery and SBC
N_SBC = 200
N_CHAINS, N_TUNE, N_DRAWS = 4, 1000, 1000
TARGET_ACCEPT = 0.95  # PyMC's default 0.8 left a few divergences when A is close to 1
THIN = 10  # thinning of posterior draws before ranking

pd.DataFrame(
    {"mu": mu, "sigma": sigma},
    index=["logit A", "logit B", "log sigma_y", "log sigma_x/sigma_y", "log block length"],
).round(3)

In [ ]:
# [rng] One root generator, one stream per purpose
rng = np.random.default_rng(ROOT_SEED)
rng_filter, rng_sim, rng_prior_pred, rng_sbc_sim, rng_sbc_fit, rng_sbc_pit = rng.spawn(6)

In [ ]:
# [plot-style] Colours and axes shared by all figures
BLUE, ORANGE, MUTED, GRID = "#2a78d6", "#eb6834", "#898781", "#e1e0d9"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": "#52514e",
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlesize": 9, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "figure.constrained_layout.use": True,
})

## [model] The model

On trial $t$ the subject has an internal state $x_t$ (how much they compensate), produces
a response $y_t$, and learns from the error they observe, $p_t - y_t$:

$$
x_{t+1} = A\, x_t + B\,(p_t - y_t) + \eta_t, \qquad y_t = x_t + \varepsilon_t,
$$

with $\eta_t \sim \mathcal N(0, \sigma_x^2)$ (state noise),
$\varepsilon_t \sim \mathcal N(0, \sigma_y^2)$ (output noise) and a first state
$x_1 \sim \mathcal N(0, \sigma_x^2)$. $A$ is retention, $B$ is the learning rate.

The priors are normal on an unbounded scale ($\operatorname{logit} A$,
$\operatorname{logit} B$, $\log \sigma_y$, $\log(\sigma_x/\sigma_y)$), with the constants
in `[config]`. The state noise is specified through its ratio to the output noise.

In [ ]:
# [sample-prior] Draw parameters from the prior
def sample_prior(rng, size):
    # Returns theta with shape (size, 4), columns (A, B, sigma_x, sigma_y).
    A = expit(rng.normal(mu_logit_A, sigma_logit_A, size))
    B = expit(rng.normal(mu_logit_B, sigma_logit_B, size))
    sigma_y = np.exp(rng.normal(mu_log_sigma_y, sigma_log_sigma_y, size))
    sigma_x = sigma_y * np.exp(rng.normal(mu_log_ratio, sigma_log_ratio, size))
    return np.column_stack([A, B, sigma_x, sigma_y])

## [schedule] Perturbation schedules

A sitting has 3 to 10 blocks. The perturbation is constant within a block. The first block
is a baseline ($p_t = 0$); at each boundary the perturbation moves to one of the other two
values, with equal probability. Block lengths are log-normal, mostly between 10 and 50
trials, rounded to whole trials.

The values are stored as `[0, +1, -1]`, so "move to one of the other two" is a step of 1
or 2 positions around that list.

In [ ]:
# [sample-schedule] Draw one perturbation schedule
def sample_schedule(rng):
    # Returns p with shape (T,): the perturbation on every trial of one sitting.
    n_blocks = rng.integers(N_BLOCKS_MIN, N_BLOCKS_MAX + 1)
    block_len = np.rint(np.exp(rng.normal(mu_log_block_len, sigma_log_block_len, n_blocks))).astype(int)
    steps = rng.integers(1, 3, n_blocks - 1)  # 1 or 2: always a change
    position = np.concatenate([[0], np.cumsum(steps)]) % 3  # first block is baseline
    return np.repeat(PERTURBATION_VALUES[position], block_len)

In [ ]:
# [simulate-sitting] The simulator: the model equations, trial by trial
def simulate_sitting(rng, theta, p, size=()):
    # Returns y with shape (*size, T): `size` independent sittings sharing theta and p.
    # The loop over trials is the recursion itself; sittings are vectorized.
    A, B, sigma_x, sigma_y = theta
    y = np.empty((*size, len(p)))
    x = rng.normal(0.0, sigma_x, size)  # x_1
    for t in range(len(p)):
        y[..., t] = x + rng.normal(0.0, sigma_y, size)
        x = A * x + B * (p[t] - y[..., t]) + rng.normal(0.0, sigma_x, size)
    return y

## [kalman] The exact likelihood: a Kalman filter

The state is hidden, but the model is linear and Gaussian, so the likelihood of a whole
sitting is available exactly. Write $m_t$ and $P_t$ for the mean and variance of $x_t$
given the earlier responses $y_{1:t-1}$, starting from $m_1 = 0$, $P_1 = \sigma_x^2$.
Then for each trial, in this order:

1. **Predict the response.** $y_t \mid y_{1:t-1} \sim \mathcal N(m_t,\ P_t + \sigma_y^2)$.
   This is the trial's contribution to the likelihood.
2. **Update on the response.** $K_t = P_t / (P_t + \sigma_y^2)$,
   $\ m_{t\mid t} = m_t + K_t (y_t - m_t)$, $\ P_{t\mid t} = (1 - K_t) P_t$.
3. **Predict the next state,** using the same $y_t$ as a known input.
   $m_{t+1} = A\, m_{t\mid t} + B\,(p_t - y_t)$, $\ P_{t+1} = A^2 P_{t\mid t} + \sigma_x^2$.

The log-likelihood of the sitting is the sum of the step-1 terms.

The filter is written once, in PyTensor, so that the same definition is checked here and
used by PyMC below. `pytensor.scan` is PyTensor's loop: it calls `kalman_step` once per
trial, carrying $m$ and $P$ forward. (PyMC's ready-made state-space models are in
`pymc-extras`, which does not support the pinned PyMC 6.3.)

In [ ]:
# [kalman-logp] The Kalman log-likelihood as a PyTensor expression
def kalman_step(p_t, y_t, m, P, A, B, var_x, var_y):
    S = P + var_y  # predictive variance of y_t
    logp_t = -0.5 * (pt.log(2 * np.pi * S) + (y_t - m) ** 2 / S)
    K = P / S
    m_filt = m + K * (y_t - m)
    P_filt = (1 - K) * P
    return A * m_filt + B * (p_t - y_t), A**2 * P_filt + var_x, logp_t


def kalman_logp(y, A, B, sigma_x, sigma_y, p):
    # Symbolic log-likelihood of one sitting: log p(y | theta, p).
    # Argument order (value first, then parameters) is the one PyMC expects of a log-density.
    _, _, logp_t = pytensor.scan(
        kalman_step,
        sequences=[p, y],
        outputs_info=[pt.zeros((), dtype="float64"), sigma_x**2, None],  # m_1, P_1, (no carry)
        non_sequences=[A, B, sigma_x**2, sigma_y**2],
        return_updates=False,
    )
    return logp_t.sum()

In [ ]:
# [kalman-compile] Compile the expression into an ordinary function of numbers
A_, B_, sigma_x_, sigma_y_ = pt.dscalars("A", "B", "sigma_x", "sigma_y")
p_, y_ = pt.dvectors("p", "y")
kalman_loglik = pytensor.function(
    [y_, A_, B_, sigma_x_, sigma_y_, p_], kalman_logp(y_, A_, B_, sigma_x_, sigma_y_, p_)
)

## [check-1] Check 1: the filter against brute force

Because every equation is linear, each response is a weighted sum of the noise terms plus
a constant. So the whole sitting $y_{1:T}$ is one multivariate Gaussian, whose mean and
covariance can be computed without any filtering.

`joint_gaussian` does this by running the model equations on *coefficients* instead of
numbers. Each quantity is a row of $2T + 1$ coefficients: its weights on
$\eta_1 \dots \eta_T$, on $\varepsilon_1 \dots \varepsilon_T$, and a constant. With $G$
the weights of the responses on the noise terms and $D$ the diagonal matrix of the noise
variances, the mean is the constant column and the covariance is $G D G^\top$.

The filter must give the same number as the density of this Gaussian. We compare them at
parameters *different* from the ones that generated the data, on short sittings
(`T_SHORT` trials, a random perturbation on every trial) and on full sittings.

In [ ]:
# [joint-gaussian] Mean and covariance of a whole sitting, without filtering
def joint_gaussian(theta, p):
    A, B, sigma_x, sigma_y = theta
    T = len(p)
    unit = np.eye(2 * T + 1)  # rows: eta_1..eta_T, eps_1..eps_T, constant
    Y = np.empty((T, 2 * T + 1))
    x = unit[0]  # x_1 = eta_1
    for t in range(T - 1):
        Y[t] = x + unit[T + t]
        x = A * x + B * (p[t] * unit[-1] - Y[t]) + unit[t + 1]
    Y[T - 1] = x + unit[2 * T - 1]
    G = Y[:, :-1]
    D = np.repeat([sigma_x**2, sigma_y**2], T)
    return Y[:, -1], (G * D) @ G.T

In [ ]:
# [check-filter] Kalman log-likelihood against the joint Gaussian density
thetas_gen = sample_prior(rng_filter, N_FILTER_CASES)
thetas_eval = sample_prior(rng_filter, N_FILTER_CASES)
rows = []
for theta_gen, theta_eval in zip(thetas_gen, thetas_eval):
    for kind, p in [
        ("short", rng_filter.choice(PERTURBATION_VALUES, T_SHORT)),
        ("full", sample_schedule(rng_filter)),
    ]:
        y = simulate_sitting(rng_filter, theta_gen, p)
        mean, cov = joint_gaussian(theta_eval, p)
        rows.append({
            "sitting": kind,
            "T": len(p),
            "kalman": float(kalman_loglik(y, *theta_eval, p)),
            "brute_force": multivariate_normal(mean, cov).logpdf(y),
        })
check_filter = pd.DataFrame(rows)
check_filter["abs_diff"] = (check_filter["kalman"] - check_filter["brute_force"]).abs()
assert check_filter["abs_diff"].max() < 1e-6

check_filter.groupby("sitting").agg(
    cases=("T", "size"), T_min=("T", "min"), T_max=("T", "max"),
    loglik_min=("kalman", "min"), loglik_max=("kalman", "max"), max_abs_diff=("abs_diff", "max"),
)

## [check-2] Check 2: the simulator against the model

The simulator and the filter are now compared through the same joint Gaussian. For
`N_SIM_CASES` parameter draws and full schedules we simulate `N_SIM` sittings each and ask
whether they look like draws from $\mathcal N(\text{mean}, \text{cov})$:

- **Mean and covariance, entry by entry.** Each sample mean and each sample covariance
  entry is turned into a z-score using its Monte Carlo standard error. If the simulator is
  right, about 4.6% of z-scores exceed 2 in size.
- **The whole distribution.** The squared Mahalanobis distance of each sitting from the
  mean should follow a $\chi^2$ distribution with $T$ degrees of freedom; a
  Kolmogorov–Smirnov (KS) test gives one p-value per case. With a correct simulator these
  p-values are themselves uniform between 0 and 1, so the smallest of `N_SIM_CASES` is
  expected to be small; what must not happen is many small ones.

In [ ]:
# [check-simulator] Simulated sittings against the joint Gaussian
rows = []
for theta in sample_prior(rng_sim, N_SIM_CASES):
    p = sample_schedule(rng_sim)
    Y = simulate_sitting(rng_sim, theta, p, size=(N_SIM,))
    mean, cov = joint_gaussian(theta, p)
    var = np.diag(cov)

    z_mean = (Y.mean(axis=0) - mean) / np.sqrt(var / N_SIM)
    z_cov = (np.cov(Y, rowvar=False) - cov) / np.sqrt((np.outer(var, var) + cov**2) / N_SIM)
    whitened = solve_triangular(np.linalg.cholesky(cov), (Y - mean).T, lower=True)
    mahalanobis_sq = (whitened**2).sum(axis=0)

    rows.append({
        "T": len(p),
        "mean: frac |z|>2": np.mean(np.abs(z_mean) > 2),
        "cov: frac |z|>2": np.mean(np.abs(z_cov) > 2),
        "KS p-value": kstest(mahalanobis_sq, chi2(len(p)).cdf).pvalue,
    })
check_sim = pd.DataFrame(rows)

pd.Series({
    "cases": N_SIM_CASES,
    "shortest sitting": check_sim["T"].min(),
    "longest sitting": check_sim["T"].max(),
    "mean: frac |z|>2, average over cases (expect 0.046)": check_sim["mean: frac |z|>2"].mean(),
    "cov: frac |z|>2, average over cases (expect 0.046)": check_sim["cov: frac |z|>2"].mean(),
    "smallest KS p-value": check_sim["KS p-value"].min(),
    "KS p-values below 0.05 (expect about 5%)": (check_sim["KS p-value"] < 0.05).mean(),
    "are the KS p-values uniform? p-value": kstest(check_sim["KS p-value"], "uniform").pvalue,
}).round(4)

## [check-3] Check 3: what the prior produces

Sittings drawn from the prior: a schedule, a parameter draw, and the simulated responses.
This is for judging by eye whether the learning curves and noise levels look like real
data. The table gives the spread of sitting lengths the schedule generator produces.

In [ ]:
# [prior-predictive] Simulated sittings from the prior
thetas = sample_prior(rng_prior_pred, N_PRIOR_PRED)
fig, axes = plt.subplots(4, 3, figsize=(11, 8.5), sharey=True)
for ax, theta in zip(axes.flat, thetas):
    p = sample_schedule(rng_prior_pred)
    y = simulate_sitting(rng_prior_pred, theta, p)
    trial = np.arange(1, len(p) + 1)
    ax.step(trial, p, where="mid", color=MUTED, lw=1.5, label="perturbation $p_t$")
    ax.plot(trial, y, ".", color=BLUE, ms=3.5, label="response $y_t$")
    ax.set_title("A={:.3f}  B={:.3f}  $\\sigma_x$={:.3f}  $\\sigma_y$={:.3f}".format(*theta))
    ax.grid(axis="y", color=GRID, lw=0.6)
for ax in axes[-1]:
    ax.set_xlabel("trial")
for ax in axes[:, 0]:
    ax.set_ylabel("perturbation units")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncols=2, frameon=False)
plt.show()

In [ ]:
# [sitting-lengths] How long are the sittings?
lengths = np.array([len(sample_schedule(rng_prior_pred)) for _ in range(5000)])
pd.Series(lengths, name="trials per sitting").describe(percentiles=[0.025, 0.25, 0.5, 0.75, 0.975]).round(0)

## [check-4] Check 4: recovery and calibration with the exact likelihood

The model is built once in PyMC, with the schedule and responses as data that can be
swapped. NUTS samples the four unbounded parameters. The sitting is an observed variable
whose log-density is the Kalman filter (`pm.CustomDist` is PyMC's way to give a variable a
hand-written density). The log-likelihood of each draw is kept as `loglik`, to serve as a
data-dependent test quantity in SBC.

**SBC.** Repeat `N_SBC` times: draw $\theta$ from the prior and a schedule, simulate a
sitting, sample the posterior, and record where the true value ranks among the posterior
draws. If prior, simulator, likelihood and sampler are mutually consistent, these ranks
are uniform. Ranks are taken on thinned draws, so that the draws are close to independent.

**Sampler setting.** With PyMC's default step-size target (`target_accept=0.8`), 33 of 200
fits had a few divergent transitions (68 of 800,000 draws; commit `fcfa94f`), all in
sittings whose true $A$ is close to 1. There the data cannot tell $A = 0.99$ from
$A = 0.999$, so the posterior of $\operatorname{logit} A$ has a long, nearly flat upper
tail, and one step size does not suit both the tail and the bulk. Smaller steps
(`TARGET_ACCEPT = 0.95`) remove them.

In [ ]:
# [exact-model] The PyMC model with the Kalman likelihood
QUANTITIES = ["logit_A", "logit_B", "log_sigma_y", "log_ratio", "log_sigma_x", "loglik"]

with pm.Model() as exact_model:
    p_data = pm.Data("p", np.zeros(1))
    y_data = pm.Data("y", np.zeros(1))

    logit_A = pm.Normal("logit_A", mu_logit_A, sigma_logit_A)
    logit_B = pm.Normal("logit_B", mu_logit_B, sigma_logit_B)
    log_sigma_y = pm.Normal("log_sigma_y", mu_log_sigma_y, sigma_log_sigma_y)
    log_ratio = pm.Normal("log_ratio", mu_log_ratio, sigma_log_ratio)
    log_sigma_x = pm.Deterministic("log_sigma_x", log_ratio + log_sigma_y)

    pm.CustomDist(
        "sitting",
        pm.math.invlogit(logit_A), pm.math.invlogit(logit_B),
        pm.math.exp(log_sigma_x), pm.math.exp(log_sigma_y),
        p_data,
        logp=kalman_logp,
        signature="(),(),(),(),(t)->(t)",  # four scalars and a schedule give one sitting
        observed=y_data,
    )

In [ ]:
# [sbc-run] Simulate sittings from the prior and sample each exact posterior
logging.getLogger("pymc").setLevel(logging.ERROR)  # diagnostics are collected below instead

sbc_theta = sample_prior(rng_sbc_sim, N_SBC)
sbc_T = np.empty(N_SBC, dtype=int)
sbc_truth = np.empty((N_SBC, len(QUANTITIES)))
sbc_draws = np.empty((N_SBC, len(QUANTITIES), N_CHAINS * N_DRAWS))
sbc_rhat = np.empty(N_SBC)
sbc_ess = np.empty(N_SBC)
sbc_divergences = np.empty(N_SBC, dtype=int)

for i, theta in enumerate(sbc_theta):
    p = sample_schedule(rng_sbc_sim)
    y = simulate_sitting(rng_sbc_sim, theta, p)
    with exact_model:
        pm.set_data({"p": p, "y": y})
        idata = pm.sample(
            draws=N_DRAWS, tune=N_TUNE, chains=N_CHAINS, target_accept=TARGET_ACCEPT,
            cores=1,  # forking after JAX is imported can deadlock (#9)
            random_seed=rng_sbc_fit, progressbar=False,
        )
        pm.compute_log_likelihood(idata, progressbar=False)
    A, B, sigma_x, sigma_y = theta
    sbc_T[i] = len(p)
    sbc_truth[i] = [
        logit(A), logit(B), np.log(sigma_y), np.log(sigma_x / sigma_y), np.log(sigma_x),
        kalman_loglik(y, *theta, p),
    ]
    posterior = idata["posterior"].to_dataset().assign(loglik=idata["log_likelihood"]["sitting"])[QUANTITIES]
    sbc_draws[i] = posterior.to_dataarray("quantity").stack(sample=("chain", "draw")).values
    sbc_rhat[i] = az.rhat(posterior).to_dataarray().max()
    sbc_ess[i] = az.ess(posterior).to_dataarray().min()
    sbc_divergences[i] = idata["sample_stats"]["diverging"].sum()

OUT_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    OUT_DIR / "exact_sbc.npz",
    root_seed=str(ROOT_SEED), git_commit=GIT_COMMIT, quantities=QUANTITIES,
    theta=sbc_theta, T=sbc_T, truth=sbc_truth, draws=sbc_draws,
    rhat=sbc_rhat, ess=sbc_ess, divergences=sbc_divergences,
)

In [ ]:
# [sbc-diagnostics] Did the sampler behave on every sitting?
pd.Series({
    "sittings": N_SBC,
    "sittings with any divergence": int((sbc_divergences > 0).sum()),
    "total divergences": int(sbc_divergences.sum()),
    "worst R-hat": sbc_rhat.max(),
    "sittings with R-hat > 1.01": int((sbc_rhat > 1.01).sum()),
    "smallest ESS": sbc_ess.min(),
    "sittings with ESS < 400": int((sbc_ess < 400).sum()),
}).round(3)

### [sbc-ranks-text] Are the ranks uniform?

Each rank is turned into a value between 0 and 1 (a PIT value; ties are broken at
random). The plots show the difference between the empirical distribution of these values
and the uniform distribution: a calibrated quantity gives a curve that wanders around
zero. The p-value in each panel is ArviZ's test of uniformity; a p-value that is not small
means no evidence of miscalibration.

In [ ]:
# [sbc-ranks] Rank uniformity for every test quantity
thinned = sbc_draws[:, :, ::THIN]
ranks = (thinned < sbc_truth[:, :, None]).sum(axis=2)
pit = (ranks + rng_sbc_pit.uniform(size=ranks.shape)) / (thinned.shape[2] + 1)

sbc_tree = xr.DataTree.from_dict({
    "prior_sbc": xr.Dataset({name: ("sitting", pit[:, j]) for j, name in enumerate(QUANTITIES)})
})
az.plot_ecdf_pit(sbc_tree, sample_dims=["sitting"], col_wrap=3, figure_kwargs={"figsize": (11, 6)})
plt.show()

### [recovery-text] Recovery, and how much the data add to the prior

The first figure shows the posterior mean against the true value, with the central 95%
interval, for each parameter on its unbounded scale.

The second figure is the direct answer to "is this parameter identified?": the posterior
standard deviation divided by the prior standard deviation, against the length of the
sitting. A value near 1 means the data added nothing to the prior; near 0, the data
determine the parameter.

In [ ]:
# [recovery] Posterior mean and 95% interval against the true value
PARAMETERS = QUANTITIES[:5]
post_mean = sbc_draws.mean(axis=2)
post_sd = sbc_draws.std(axis=2)
post_lo, post_hi = np.quantile(sbc_draws, [0.025, 0.975], axis=2)

fig, axes = plt.subplots(1, 5, figsize=(15, 3.4))
for j, (ax, name) in enumerate(zip(axes, PARAMETERS)):
    ax.vlines(sbc_truth[:, j], post_lo[:, j], post_hi[:, j], color=BLUE, lw=0.6, alpha=0.5)
    ax.plot(sbc_truth[:, j], post_mean[:, j], ".", color=BLUE, ms=4)
    ax.axline((sbc_truth[:, j].mean(), sbc_truth[:, j].mean()), slope=1, color=MUTED, lw=1)
    ax.set_title(name)
    ax.set_xlabel("true value")
    ax.grid(color=GRID, lw=0.6)
axes[0].set_ylabel("posterior mean and 95% interval")
plt.show()

In [ ]:
# [contraction] Posterior sd relative to prior sd, against sitting length
prior_sd = np.array([
    sigma_logit_A, sigma_logit_B, sigma_log_sigma_y, sigma_log_ratio,
    np.hypot(sigma_log_sigma_y, sigma_log_ratio),
])
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2), sharey=True)
for j, (ax, name) in enumerate(zip(axes, PARAMETERS)):
    ax.plot(sbc_T, post_sd[:, j] / prior_sd[j], ".", color=BLUE, ms=4)
    ax.axhline(1.0, color=MUTED, lw=1)
    ax.set_title(name)
    ax.set_xlabel("trials in the sitting")
    ax.set_ylim(0, None)
    ax.grid(color=GRID, lw=0.6)
axes[0].set_ylabel("posterior sd / prior sd")
plt.show()

In [ ]:
# [coverage] How often the 95% interval contains the true value, by sitting length
covered = (post_lo[:, :5] <= sbc_truth[:, :5]) & (sbc_truth[:, :5] <= post_hi[:, :5])
length_group = pd.qcut(sbc_T, 3, labels=["short", "medium", "long"])
table = pd.DataFrame(covered, columns=PARAMETERS).groupby(np.asarray(length_group)).mean()
table.loc["all"] = covered.mean(axis=0)
table.insert(0, "sittings", pd.Series(length_group).value_counts().reindex(table.index).fillna(N_SBC).astype(int))
table.insert(1, "trials", pd.Series(sbc_T).groupby(np.asarray(length_group)).agg(lambda s: f"{s.min()}-{s.max()}").reindex(table.index).fillna(f"{sbc_T.min()}-{sbc_T.max()}"))
table.loc[["short", "medium", "long", "all"]].round(2)

In [ ]:
# [contraction-table] Median posterior sd / prior sd, by sitting length
table = pd.DataFrame(post_sd[:, :5] / prior_sd, columns=PARAMETERS).groupby(np.asarray(length_group)).median()
table.loc["all"] = np.median(post_sd[:, :5] / prior_sd, axis=0)
table.loc[["short", "medium", "long", "all"]].round(2)